<a href="https://colab.research.google.com/github/sowrish1004/LangChain-Tools/blob/main/LangChain_ToolCalling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
from google.colab import userdata

# Safely pull your free Gemini API key from Colab secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [3]:
!pip install -q langchain-google-genai langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.9 MB/s eta 0:00:00


In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [5]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [6]:
print(multiply.invoke({'a':3, 'b':4}))

12


# tool binding

In [158]:
llm = ChatGoogleGenerativeAI(model = 'gemini-2.5-flash')

In [160]:
llm.invoke("hi")

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d88b4-1bcf-7e80-85b9-bba4406053a1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 290, 'total_tokens': 292, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 280}})

In [161]:
llm_with_tools = llm.bind_tools([multiply])

In [162]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="I'm doing well, thank you! How can I help you today?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d88b4-6a4c-7693-bf8a-b98d649fe7b8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 16, 'total_tokens': 74, 'input_token_details': {'cache_read': 0}})

In [163]:
query = HumanMessage('can you multiply 3 with 1000')

In [164]:
messages = [query]

In [148]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [165]:
result = llm_with_tools.invoke(messages)

In [150]:
messages.append(result)

In [151]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 3, "b": 1000}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d88b1-9831-7612-b3eb-f66bd41e5a81-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': '2dcd6769-b9ec-4c2d-9ab6-dd6f66446135', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 65, 'output_tokens': 21, 'total_tokens': 86, 'input_token_details': {'cache_read': 0}})]

In [152]:
tool_result = multiply.invoke(result.tool_calls[0])

In [153]:
messages.append(tool_result)

In [154]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 3, "b": 1000}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d88b1-9831-7612-b3eb-f66bd41e5a81-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': '2dcd6769-b9ec-4c2d-9ab6-dd6f66446135', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 65, 'output_tokens': 21, 'total_tokens': 86, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='3000', name='multiply', tool_call_id='2dcd6769-b9ec-4c2d-9ab6-dd6f66446135')]

In [155]:
final_result = llm_with_tools.invoke(messages)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 14.28874859s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerMo

In [64]:
print(final_result)

content='' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019d87f3-78ae-7ae2-945c-b8fe2ee94b35-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 102, 'output_tokens': 0, 'total_tokens': 102, 'input_token_details': {'cache_read': 0}}


# Create Tool

In [126]:
# tool create

from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/7665b924ea89b212a37638d0/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate

In [127]:
get_conversion_factor.invoke({'base_currency':'USD', 'target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1776038401,
 'time_last_update_utc': 'Mon, 13 Apr 2026 00:00:01 +0000',
 'time_next_update_unix': 1776124801,
 'time_next_update_utc': 'Tue, 14 Apr 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 93.191}

In [128]:
convert.invoke({'base_currency_value':10, 'conversion_rate':91.45})

914.5

In [129]:
# Tool BINDING
llm = ChatGoogleGenerativeAI(model = 'gemini-2.5-flash-lite')

In [130]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [131]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr')]

In [132]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={})]

In [133]:
ai_message = llm_with_tools.invoke(messages)

In [134]:
messages.append(ai_message)

In [135]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'a9fac71f-a491-44e6-a0ab-b7d520f5ca2b',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': '2b7e7064-bc82-4735-bf14-57735f1ba8dd',
  'type': 'tool_call'}]

In [136]:
import json

for tool_call in ai_message.tool_calls:
  # Execute first tool and get value of conversion rate
  if tool_call['name']=='get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    print(tool_message1)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # Execute 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current argument
    tool_call['args']['conversion_rate']=conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)
  #print(tool_call)

content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1776038401, "time_last_update_utc": "Mon, 13 Apr 2026 00:00:01 +0000", "time_next_update_unix": 1776124801, "time_next_update_utc": "Tue, 14 Apr 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 93.191}' name='get_conversion_factor' tool_call_id='a9fac71f-a491-44e6-a0ab-b7d520f5ca2b'


In [139]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'convert', 'arguments': '{"base_currency_value": 10}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d889e-1e86-70c1-ada4-5de77a24fb6f-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'a9fac71f-a491-44e6-a0ab-b7d520f5ca2b', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10, 'conversion_rate': 93.191}, 'id': '2b7e7064-bc82-4735-bf14-57735f1ba8dd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 150, 'output_tokens': 44, 'total_tokens': 194, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='{"result": "succes

In [140]:
llm_with_tools.invoke(messages)

AIMessage(content='The conversion factor between USD and INR is 1 to 93.191. 10 USD is equal to 931.91 INR.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d88b0-6eec-7d53-94b4-3146ac9ef43d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 434, 'output_tokens': 35, 'total_tokens': 469, 'input_token_details': {'cache_read': 0}})